# 📰 Periódico Inteligente con Azure AI

## Descripción del proyecto

Este proyecto automatiza la lectura y análisis del periódico **El País** combinando varios servicios de Azure AI con Python.

El flujo completo es el siguiente:

1. **Scraping** — Extraemos las noticias de portada de elpais.com
2. **Azure Language Service** — Analizamos cada noticia: detectamos el idioma, extraemos ideas clave y clasificamos por categoría
3. **Clasificación de ficheros** — Guardamos cada artículo en una carpeta según su categoría e idioma
4. **Azure Speech Service** — Generamos un resumen del día en formato audio (podcast)
5. **App Streamlit** — Interfaz web que muestra las noticias analizadas y reproduce el podcast

### Servicios Azure utilizados
| Servicio | Para qué lo usamos |
|---|---|
| Azure Language Service | Detección de idioma, extracción de frases clave, análisis de sentimiento |
| Azure Speech Service | Síntesis de voz (Text-to-Speech) para el podcast diario |
| Azure Computer Vision | Alternativa OCR si el scraping no está disponible |

> **Nota sobre Computer Vision:** Aunque el scraping directo funciona correctamente, incluimos Computer Vision como alternativa robusta. Si el periódico bloqueara el scraper o cambiara su estructura HTML, se podría hacer una captura de pantalla de la portada y extraer el texto mediante OCR. Esto demuestra una arquitectura más resiliente.

## Paso 0 — Instalación de dependencias

Ejecuta esta celda solo la primera vez.

In [1]:
%pip install -r ../requirements.txt

  Using cached requests-2.31.0-py3-none-any.whl.metadata (4.6 kB)
  Using cached beautifulsoup4-4.12.3-py3-none-any.whl.metadata (3.8 kB)
  Using cached azure_ai_textanalytics-5.3.0-py3-none-any.whl.metadata (82 kB)
  Using cached azure_cognitiveservices_speech-1.38.0-py3-none-win_amd64.whl.metadata (1.5 kB)
  Using cached azure_ai_vision_imageanalysis-1.0.0-py3-none-any.whl.metadata (22 kB)
  Using cached python_dotenv-1.0.1-py3-none-any.whl.metadata (23 kB)
  Using cached streamlit-1.35.0-py2.py3-none-any.whl.metadata (8.5 kB)
  Using cached newsapi_python-0.2.7-py2.py3-none-any.whl.metadata (1.2 kB)
  Using cached altair-5.5.0-py3-none-any.whl.metadata (11 kB)
  Using cached cachetools-5.5.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached numpy-1.26.4.tar.gz (15.8 MB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'


  error: subprocess-exited-with-error
  
  × Preparing metadata (pyproject.toml) did not run successfully.
  │ exit code: 2
  ╰─> [65 lines of output]
      + c:\Users\ranger\anaconda3\envs\azureCognitive\python.exe C:\Users\ranger\AppData\Local\Temp\pip-install-vqg5wisi\numpy_62dc970908354924a183c4bdbe3e8bce\vendored-meson\meson\meson.py setup C:\Users\ranger\AppData\Local\Temp\pip-install-vqg5wisi\numpy_62dc970908354924a183c4bdbe3e8bce C:\Users\ranger\AppData\Local\Temp\pip-install-vqg5wisi\numpy_62dc970908354924a183c4bdbe3e8bce\.mesonpy-ug4210qs -Dbuildtype=release -Db_ndebug=if-release -Db_vscrt=md --native-file=C:\Users\ranger\AppData\Local\Temp\pip-install-vqg5wisi\numpy_62dc970908354924a183c4bdbe3e8bce\.mesonpy-ug4210qs\meson-python-native-file.ini
      The Meson build system
      Version: 1.2.99
      Source dir: C:\Users\ranger\AppData\Local\Temp\pip-install-vqg5wisi\numpy_62dc970908354924a183c4bdbe3e8bce
      Build dir: C:\Users\ranger\AppData\Local\Temp\pip-install-vqg5wi

## Paso 1 — Configuración de credenciales Azure

Las claves se cargan desde el fichero `.env` en la raíz del proyecto usando `python-dotenv`.

In [2]:
import os
from dotenv import load_dotenv
from pathlib import Path

# Carga el .env desde la raíz del proyecto (un nivel arriba de notebooks/)
load_dotenv(Path("..") / ".env")

LANGUAGE_KEY      = os.getenv("LANGUAGE_KEY")
LANGUAGE_ENDPOINT = os.getenv("LANGUAGE_ENDPOINT")
SPEECH_KEY        = os.getenv("SPEECH_KEY")
SPEECH_REGION     = os.getenv("SPEECH_REGION")
VISION_KEY        = os.getenv("VISION_KEY")
VISION_ENDPOINT   = os.getenv("VISION_ENDPOINT")
NEWSAPI_KEY       = os.getenv("NEWSAPI_KEY")
NEWSDATA_KEY      = os.getenv("NEWSDATA_KEY")
TRANSLATOR_KEY    = os.getenv("TRANSLATOR_KEY")
TRANSLATOR_REGION = os.getenv("TRANSLATOR_REGION")
TRANSLATOR_ENDPOINT = os.getenv("TRANSLATOR_ENDPOINT")

# Verificar que las claves están cargadas (muestra solo si están presentes, no el valor)
claves = {
    "LANGUAGE_KEY": LANGUAGE_KEY,
    "LANGUAGE_ENDPOINT": LANGUAGE_ENDPOINT,
    "SPEECH_KEY": SPEECH_KEY,
    "SPEECH_REGION": SPEECH_REGION,
    "VISION_KEY": VISION_KEY,
    "VISION_ENDPOINT": VISION_ENDPOINT,
    "NEWSAPI_KEY": NEWSAPI_KEY,
    "NEWSDATA_KEY": NEWSDATA_KEY,
    "TRANSLATOR_KEY": TRANSLATOR_KEY,
    "TRANSLATOR_REGION": TRANSLATOR_REGION,
    "TRANSLATOR_ENDPOINT": TRANSLATOR_ENDPOINT,
}

for nombre, valor in claves.items():
    estado = "✓ cargada" if valor and "TU_" not in valor else "⚠️  pendiente de rellenar"
    print(f"{nombre}: {estado}")

LANGUAGE_KEY: ✓ cargada
LANGUAGE_ENDPOINT: ✓ cargada
SPEECH_KEY: ✓ cargada
SPEECH_REGION: ✓ cargada
VISION_KEY: ⚠️  pendiente de rellenar
VISION_ENDPOINT: ⚠️  pendiente de rellenar
NEWSAPI_KEY: ✓ cargada
NEWSDATA_KEY: ✓ cargada
TRANSLATOR_KEY: ✓ cargada
TRANSLATOR_REGION: ✓ cargada
TRANSLATOR_ENDPOINT: ✓ cargada


## Paso 2 — Scraping de El País

Usamos `requests` y `BeautifulSoup` para extraer las noticias de la portada. El módulo `src/scraper_elpais.py` contiene las funciones reutilizables.

In [3]:
import sys
import json

# Añadir la raíz del proyecto al path para poder importar desde src/
sys.path.append(str(Path("..")))

from src.scraper_elpais import obtener_noticias_portada, guardar_noticias

print("Scrapeando portada de El País...\n")
noticias = obtener_noticias_portada(max_noticias=10)

print(f"✓ {len(noticias)} noticias obtenidas\n")
print("-" * 60)

for i, n in enumerate(noticias, 1):
    print(f"{i}. [{n['seccion'].upper()}] {n['titulo']}")
    if n['resumen']:
        print(f"   {n['resumen'][:100]}...")
    print()

Scrapeando portada de El País...

✓ 10 noticias obtenidas

------------------------------------------------------------
1. [INTERNACIONAL] Irán rechaza la propuesta de Trump para poner fin a la guerra y reivindica su soberanía sobre Ormuz
   Los países mediadores intentan lograr una reunión entre los dos adversarios antes de que acabe la se...

2. [INTERNACIONAL] EE UU amenaza con atacar Irán con más intensidad si Teherán no acepta que ha perdido la guerra

3. [ESPANA] La mayoría del Congreso rechaza la guerra pese a la bronca política
   Sánchez trata de ridiculizar los conocimientos de Feijóo y este dice que en Europa lo tildan de “tra...

4. [ESPANA] El último control a Montero en el Congreso anticipa la tensión de la campaña andaluza

5. [ESPANA] PP y Vox avanzan en un acuerdo programático para un Gobierno de coalición en Extremadura
   Los negociadores de los dos partidos mantienen un encuentro en Mérida...

6. [TECNOLOGIA] Meta y YouTube pierden en EE UU el juicio sobre la adicci

### Guardar las noticias en JSON

In [4]:
guardar_noticias(noticias, ruta="../data/noticias_hoy.json")

# Mostrar el JSON de la primera noticia como ejemplo
print("\nEjemplo — primera noticia:")
print(json.dumps(noticias[0], ensure_ascii=False, indent=2))

✓ 10 noticias guardadas en ../data/noticias_hoy.json

Ejemplo — primera noticia:
{
  "titulo": "Irán rechaza la propuesta de Trump para poner fin a la guerra y reivindica su soberanía sobre Ormuz",
  "seccion": "internacional",
  "resumen": "Los países mediadores intentan lograr una reunión entre los dos adversarios antes de que acabe la semana y advierten que puede ser la última oportunidad",
  "url": "https://elpais.com/internacional/2026-03-25/iran-rechaza-la-oferta-de-estados-unidos-para-acabar-la-guerra.html",
  "fecha_scraping": "2026-03-26 01:07",
  "fuente": "elpais"
}


---

## Paso 3 — Azure Language Service (PLN)

Ahora mandamos el texto de cada noticia a **Azure Language Service** para extraer tres tipos de información:

| Análisis | Qué nos devuelve | Para qué lo usamos |
|---|---|---|
| Detección de idioma | Español, Inglés... + % confianza | Clasificar por carpeta de idioma |
| Extracción de frases clave | Lista de conceptos importantes | Resumen del día y podcast |
| Análisis de sentimiento | Positivo / Negativo / Neutro / Mixto | Dashboard visual en Streamlit |

> La API acepta hasta **10 documentos por llamada** en el tier gratuito. Nuestro módulo ya gestiona esto automáticamente.

### 3.1 — Crear el cliente de Azure Language

In [5]:
from src.azure_language import crear_cliente

# Verificar que las claves están cargadas antes de continuar
if not LANGUAGE_KEY or "TU_" in LANGUAGE_KEY:
    raise ValueError("⚠️  Rellena LANGUAGE_KEY en el fichero .env antes de continuar")

cliente_language = crear_cliente(
    endpoint=LANGUAGE_ENDPOINT,
    key=LANGUAGE_KEY
)

print("✓ Cliente Azure Language creado correctamente")

✓ Cliente Azure Language creado correctamente


### 3.2 — Analizar todas las noticias

Esta celda manda todas las noticias a Azure y enriquece cada una con idioma, frases clave, sentimiento y categoría inferida.

In [6]:
from src.azure_language import analizar_noticias

print("Analizando noticias con Azure Language Service...\n")
noticias_analizadas = analizar_noticias(cliente_language, noticias)
print(f"\n✓ {len(noticias_analizadas)} noticias analizadas")

Analizando noticias con Azure Language Service...

  → 10 documentos en 1 lote(s) de máx. 10
  → Detectando idiomas...
  → Extrayendo frases clave...
  → Analizando sentimiento...

✓ 10 noticias analizadas


### 3.3 — Ver resultados del análisis

In [7]:
print(f"{'#':<3} {'CATEGORÍA':<12} {'IDIOMA':<10} {'SENTIMIENTO':<10} TÍTULO")
print("-" * 80)

for i, n in enumerate(noticias_analizadas, 1):
    print(f"{i:<3} {n['categoria'].upper():<12} {n['codigo_idioma']:<10} {n['sentimiento']:<10} {n['titulo'][:50]}")

#   CATEGORÍA    IDIOMA     SENTIMIENTO TÍTULO
--------------------------------------------------------------------------------
1   OTROS        es         negative   Irán rechaza la propuesta de Trump para poner fin 
2   OTROS        es         negative   EE UU amenaza con atacar Irán con más intensidad s
3   POLITICA     es         negative   La mayoría del Congreso rechaza la guerra pese a l
4   POLITICA     es         neutral    El último control a Montero en el Congreso anticip
5   DEPORTES     es         neutral    PP y Vox avanzan en un acuerdo programático para u
6   TECNOLOGIA   es         negative   Meta y YouTube pierden en EE UU el juicio sobre la
7   OTROS        es         neutral    Trinidad fue a visitar a su marido a la cárcel, un
8   OTROS        es         negative   Noelia, antes de su eutanasia: “No quiero ser ejem
9   POLITICA     es         negative   Meloni fuerza la dimisión de una ministra que se n
10  POLITICA     es         neutral    El consejo de Indra se 

In [8]:
# Ver el análisis completo de una noticia concreta (cambia el índice)
INDICE = 0

n = noticias_analizadas[INDICE]
print(f"Título    : {n['titulo']}")
print(f"Categoría : {n['categoria']}")
print(f"Idioma    : {n['idioma']} ({n['codigo_idioma']})")
print(f"Sentimiento: {n['sentimiento']}")
print(f"  · Positivo: {n['score_sentimiento']['positivo']}")
print(f"  · Neutro  : {n['score_sentimiento']['neutro']}")
print(f"  · Negativo: {n['score_sentimiento']['negativo']}")
print(f"Frases clave:")
for frase in n['frases_clave']:
    print(f"  · {frase}")

Título    : Irán rechaza la propuesta de Trump para poner fin a la guerra y reivindica su soberanía sobre Ormuz
Categoría : otros
Idioma    : Spanish (es)
Sentimiento: negative
  · Positivo: 0.0
  · Neutro  : 0.43
  · Negativo: 0.56
Frases clave:
  · países mediadores
  · Irán
  · soberanía
  · reunión
  · dos adversarios
  · última oportunidad
  · propuesta
  · Trump
  · fin
  · guerra
  · Ormuz.
  · semana


### 3.4 — Guardar noticias analizadas en JSON

In [9]:
import json
from pathlib import Path

ruta = Path("../data/noticias_analizadas.json")
ruta.parent.mkdir(parents=True, exist_ok=True)

with open(ruta, "w", encoding="utf-8") as f:
    json.dump(noticias_analizadas, f, ensure_ascii=False, indent=2)

print(f"✓ Guardado en {ruta}")

✓ Guardado en ..\data\noticias_analizadas.json


---

## Paso 4 — Clasificación automática de artículos en carpetas

Con el análisis de Azure ya hecho, clasificamos cada artículo guardándolo como fichero `.json` en una carpeta según su **categoría** e **idioma**.

La estructura resultante será:
```
data/articulos/
├── politica/
│   ├── es/
│   └── en/
├── economia/
│   └── es/
├── deportes/
│   └── es/
...
```

### 4.1 — Crear la estructura de carpetas

In [11]:
from src.clasificador import crear_estructura_carpetas
from pathlib import Path

BASE = Path("../data/articulos")
carpetas = crear_estructura_carpetas(base=BASE)

print(f"✓ {len(carpetas)} carpetas creadas o verificadas")
print("\nEstructura:")
for c in carpetas:
    print(f"  {c}")

✓ 18 carpetas creadas o verificadas

Estructura:
  ..\data\articulos\deportes\es
  ..\data\articulos\deportes\en
  ..\data\articulos\deportes\otros
  ..\data\articulos\politica\es
  ..\data\articulos\politica\en
  ..\data\articulos\politica\otros
  ..\data\articulos\economia\es
  ..\data\articulos\economia\en
  ..\data\articulos\economia\otros
  ..\data\articulos\tecnologia\es
  ..\data\articulos\tecnologia\en
  ..\data\articulos\tecnologia\otros
  ..\data\articulos\cultura\es
  ..\data\articulos\cultura\en
  ..\data\articulos\cultura\otros
  ..\data\articulos\otros\es
  ..\data\articulos\otros\en
  ..\data\articulos\otros\otros


### 4.2 — Clasificar y guardar los artículos

In [12]:
from src.clasificador import clasificar_noticias

print("Clasificando artículos...\n")
resultado = clasificar_noticias(noticias_analizadas, base=BASE)

print(f"✓ {resultado['total']} artículos clasificados\n")
print("Distribución por carpeta:")
for carpeta, cantidad in sorted(resultado['distribucion'].items()):
    print(f"  {carpeta:<25} → {cantidad} artículo(s)")

Clasificando artículos...

✓ 10 artículos clasificados

Distribución por carpeta:
  deportes/es               → 1 artículo(s)
  otros/es                  → 4 artículo(s)
  politica/es               → 4 artículo(s)
  tecnologia/es             → 1 artículo(s)


### 4.3 — Inventario de artículos guardados

In [13]:
from src.clasificador import listar_articulos

inventario = listar_articulos(base=BASE)

print("Inventario de artículos guardados:\n")
total = 0
for categoria, idiomas in inventario.items():
    for idioma, cantidad in idiomas.items():
        print(f"  [{categoria.upper():<12}] [{idioma}]  →  {cantidad} artículo(s)")
        total += cantidad

print(f"\n  Total: {total} artículo(s) en disco")

Inventario de artículos guardados:

  [DEPORTES    ] [es]  →  9 artículo(s)
  [ECONOMIA    ] [es]  →  4 artículo(s)
  [OTROS       ] [es]  →  19 artículo(s)
  [POLITICA    ] [es]  →  23 artículo(s)
  [TECNOLOGIA  ] [es]  →  5 artículo(s)

  Total: 60 artículo(s) en disco


### 4.4 — Leer un artículo clasificado como ejemplo

In [14]:
import json

ficheros = list(BASE.rglob("*.json"))

if ficheros:
    with open(ficheros[0], encoding="utf-8") as f:
        articulo = json.load(f)

    print(f"Fichero  : {ficheros[0]}")
    print(f"Título   : {articulo['titulo']}")
    print(f"Categoría: {articulo['categoria']}")
    print(f"Idioma   : {articulo['idioma']}")
    print(f"Sentimiento: {articulo['sentimiento']}")
    print(f"Frases clave: {', '.join(articulo['frases_clave'][:5])}")
else:
    print("No se encontraron ficheros. Revisa el paso 4.2")

Fichero  : ..\data\articulos\otros\es\2026-03-25_16-19_Irán rechaza la propuesta de Trump para poner fin a la guerr.json
Título   : Irán rechaza la propuesta de Trump para poner fin a la guerra y reivindica su soberanía sobre Ormuz
Categoría: otros
Idioma   : Spanish
Sentimiento: negative
Frases clave: Irán, soberanía, Teherán, EE UU, propuesta


---

## Paso 5 — Azure Speech: generación del podcast

En este paso convertimos las noticias del día en un **archivo de audio MP3** usando Azure Speech Service.

El proceso tiene dos fases:
1. **Construir el texto narrativo** — no mandamos los titulares en bruto, sino un texto redactado como presentador de radio, con introducción, bloques por categoría y cierre
2. **Sintetizar la voz** — Azure Speech convierte ese texto a audio con la voz neuronal `es-ES-ElviraNeural`

El resultado se guarda en `output/podcast_YYYY-MM-DD.mp3`

### 5.1 — Previsualizar el texto del podcast

Antes de llamar a la API y consumir créditos, revisamos el texto que se va a narrar.

In [15]:
from src.azure_speech import previsualizar_texto_podcast

texto_podcast = previsualizar_texto_podcast(noticias_analizadas)

print(f"Longitud del texto: {len(texto_podcast)} caracteres\n")
print("-" * 60)
print(texto_podcast)

Longitud del texto: 1252 caracteres

------------------------------------------------------------
Bienvenidos al resumen del día. Hoy es jueves 26 de March de 2026. A continuación, las noticias más importantes. En Política. La mayoría del Congreso rechaza la guerra pese a la bronca política. También, El último control a Montero en el Congreso anticipa la tensión de la campaña andaluza. Además, Meloni fuerza la dimisión de una ministra que se negaba a renunciar. Y para cerrar esta sección, El consejo de Indra se da una tregua y Escribano resiste el pulso a la SEPI. En Deportes. PP y Vox avanzan en un acuerdo programático para un Gobierno de coalición en Extremadura. En Tecnología. Meta y YouTube pierden en EE UU el juicio sobre la adicción de los menores a las redes sociales. En Otras noticias. Irán rechaza la propuesta de Trump para poner fin a la guerra y reivindica su soberanía sobre Ormuz. También, EE UU amenaza con atacar Irán con más intensidad si Teherán no acepta que ha perdido 

### 5.2 — Generar el audio con Azure Speech

In [16]:
from src.azure_speech import generar_podcast

if not SPEECH_KEY or "TU_" in SPEECH_KEY:
    raise ValueError("⚠️  Rellena SPEECH_KEY en el fichero .env antes de continuar")

print("Generando podcast...\n")
ruta_audio = generar_podcast(
    noticias_analizadas=noticias_analizadas,
    speech_key=SPEECH_KEY,
    speech_region=SPEECH_REGION,
    carpeta_salida="../output"
)

print(f"\n✓ Podcast guardado en: {ruta_audio}")

Generando podcast...

  → Construyendo texto del podcast...
  → Texto generado (1252 caracteres)
  → Convirtiendo a voz con Azure Speech...
  ✓ Audio generado: ..\output\podcast_2026-03-26.mp3

✓ Podcast guardado en: ..\output\podcast_2026-03-26.mp3


### 5.3 — Reproducir el audio en el notebook

In [17]:
import IPython.display as ipd

print(f"Reproduciendo: {ruta_audio}")
ipd.display(ipd.Audio(ruta_audio))

Reproduciendo: ..\output\podcast_2026-03-26.mp3


---

## Paso 6 — Fuentes adicionales de noticias

Además de El País, el proyecto recoge noticias de tres fuentes más:

| Fuente | Tipo | Idioma | Módulo |
|---|---|---|---|
| **20 Minutos** | Scraping web | Español | `src/scraper_20minutos.py` |
| **NewsData.io** | API REST | Inglés (BBC, NYT...) | `src/api_newsdata.py` |
| **NewsAPI.org** | API REST (`/v2/everything`) | Español | `src/api_newsapi.py` |

Todas devuelven la misma estructura unificada: `{titulo, seccion, resumen, url, fecha_scraping, fuente}`.

### 6.1 — Scraping de 20 Minutos

In [18]:
from src.scraper_20minutos import obtener_noticias_portada as obtener_20min

print("Scrapeando portada de 20 Minutos...\n")
noticias_20min = obtener_20min(max_noticias=10)

print(f"✓ {len(noticias_20min)} noticias obtenidas de 20 Minutos\n")
print("-" * 60)
for i, n in enumerate(noticias_20min, 1):
    print(f"{i}. [{n['seccion'].upper()}] {n['titulo']}")
    if n['resumen']:
        print(f"   {n['resumen'][:100]}...")
    print()

Scrapeando portada de 20 Minutos...

✓ 10 noticias obtenidas de 20 Minutos

------------------------------------------------------------
1. [POLITICA] Irán rechaza el plan de Trump para poner fin a la guerra y EEUU le amenaza con "desatar el infierno"

2. [POLITICA] EEUU trabaja para lograr una reunión entre Vance y el régimen iraní este fin de semana

3. [OTROS] El gráfico que muestra laenorme y sospechosa inversiónen el mercado de petróleo minutos antes del anuncio del plan de paz

4. [OTROS] Sánchez se reivindica frente a Aznar porla guerray Feijóo ve "insuficiente" el plan de ayudas para la crisis

5. [OTROS] Los audios del momento del histórico apagón:"Hostia, hostia, ¡a tomar por culo! Nos estamos desconectando"

6. [CULTURA] Elduro testimonio de la joven Noelia Castilloantes de recibir la eutanasia: "Por fin lo he conseguido"

7. [OTROS] La jovenrecibirá este jueves la muerte asistidatras meses de batalla judicial con su padre, que se opone

8. [OTROS] Rosalía, obligada a abando

### 6.2 — API de NewsData.io

Trae noticias internacionales en inglés (BBC, NYT, etc.) usando la API de NewsData.io.

In [19]:
from src.api_newsdata import obtener_noticias_newsdata

if not NEWSDATA_KEY:
    print("⚠️  NEWSDATA_KEY no encontrada en .env")
else:
    print("Consultando NewsData.io...\n")
    noticias_newsdata = obtener_noticias_newsdata(api_key=NEWSDATA_KEY, max_noticias=10)

    print(f"✓ {len(noticias_newsdata)} noticias obtenidas de NewsData.io\n")
    print("-" * 60)
    for i, n in enumerate(noticias_newsdata, 1):
        print(f"{i}. [{n['seccion'].upper()}] {n['titulo'][:80]}")
        print()

Consultando NewsData.io...

✓ 10 noticias obtenidas de NewsData.io

------------------------------------------------------------
1. [ECONOMIA] CTA Construction Managers Awarded Contract to Build New Burlington Police Statio

2. [ECONOMIA] Better Industrial Stock: Tesla vs. Rocket Lab

3. [ECONOMIA] How Vingroup and VinFast Are Building a Green-Tech Ecosystem for Global Expansio

4. [OTROS] 2027: I will help Peter Obi get LP ticket if he loses in ADC – Datti Baba-Ahmed

5. [OTROS] Bus service to divert route due to road closure

6. [ECONOMIA] News24 | SA can find the money to cut fuel levies by 50% for 6 months – DA

7. [OTROS] News24 | National police commissioner Fannie Masemola to be charged over ‘Cat’ M

8. [TECNOLOGIA] How Vingroup and VinFast Are Building a Green-Tech Ecosystem for Global Expansio

9. [ECONOMIA] German Ifo Index Takes A Nose Dive In March

10. [ECONOMIA] CTA Construction Managers Awarded Contract to Build New Burlington Police Statio



### 6.3 — API de NewsAPI.org

Trae noticias en español usando el endpoint `/v2/everything` de NewsAPI.org.

In [20]:
from src.api_newsapi import obtener_noticias_newsapi

if not NEWSAPI_KEY:
    print("⚠️  NEWSAPI_KEY no encontrada en .env")
else:
    print("Consultando NewsAPI.org...\n")
    noticias_newsapi = obtener_noticias_newsapi(api_key=NEWSAPI_KEY, max_noticias=10)

    print(f"✓ {len(noticias_newsapi)} noticias obtenidas de NewsAPI.org\n")
    print("-" * 60)
    for i, n in enumerate(noticias_newsapi, 1):
        print(f"{i}. [{n['seccion'].upper()}] {n['titulo'][:80]}")
        print()

Consultando NewsAPI.org...

✓ 3 noticias obtenidas de NewsAPI.org

------------------------------------------------------------
1. [CULTURA] México dicta sentencia de Proyecto fin del mundo: simplemente, una de las mejore

2. [OTROS] Mhoni Vidente: Horóscopos y PREDICCIONES de HOY MIÉRCOLES 25 de marzo para tu si

3. [OTROS] Horóscopo Chino: Conoce qué es lo que te depara tu signo oriental para este miér



### 6.4 — Resumen de todas las fuentes

In [21]:
# Combinar todas las fuentes
todas_las_noticias = noticias.copy()  # El País (paso 2)
todas_las_noticias += noticias_20min
todas_las_noticias += noticias_newsdata if 'noticias_newsdata' in dir() else []
todas_las_noticias += noticias_newsapi if 'noticias_newsapi' in dir() else []

# Resumen por fuente
from collections import Counter
conteo = Counter(n.get("fuente", "elpais") for n in todas_las_noticias)

print(f"Total de noticias recopiladas: {len(todas_las_noticias)}\n")
print("Por fuente:")
for fuente, cantidad in conteo.most_common():
    print(f"  {fuente:<15} → {cantidad} noticias")

Total de noticias recopiladas: 33

Por fuente:
  elpais          → 10 noticias
  20minutos       → 10 noticias
  newsdata        → 10 noticias
  newsapi         → 3 noticias


### 6.5 — Analizar las noticias adicionales con Azure Language

Las noticias de 20 Minutos, NewsData y NewsAPI aún no tienen los campos de análisis (`categoria`, `codigo_idioma`, `sentimiento`, `frases_clave`). Las pasamos por Azure Language Service igual que hicimos en el paso 3 con El País.

In [22]:
# Noticias adicionales = todas menos El País (ya analizadas en paso 3)
noticias_adicionales = []
noticias_adicionales += noticias_20min
noticias_adicionales += noticias_newsdata if 'noticias_newsdata' in dir() else []
noticias_adicionales += noticias_newsapi if 'noticias_newsapi' in dir() else []

print(f"Noticias adicionales a analizar: {len(noticias_adicionales)}\n")
print("Analizando con Azure Language Service...\n")

noticias_adicionales_analizadas = analizar_noticias(cliente_language, noticias_adicionales)

print(f"\n✓ {len(noticias_adicionales_analizadas)} noticias adicionales analizadas")

# Mostrar resultados
print(f"\n{'#':<3} {'FUENTE':<12} {'CATEGORÍA':<12} {'IDIOMA':<6} {'SENTIMIENTO':<10} TÍTULO")
print("-" * 90)
for i, n in enumerate(noticias_adicionales_analizadas, 1):
    print(f"{i:<3} {n.get('fuente','?'):<12} {n['categoria'].upper():<12} {n['codigo_idioma']:<6} {n['sentimiento']:<10} {n['titulo'][:40]}")

Noticias adicionales a analizar: 23

Analizando con Azure Language Service...

  → 23 documentos en 3 lote(s) de máx. 10
  → Detectando idiomas...
  → Extrayendo frases clave...
  → Analizando sentimiento...

✓ 23 noticias adicionales analizadas

#   FUENTE       CATEGORÍA    IDIOMA SENTIMIENTO TÍTULO
------------------------------------------------------------------------------------------
1   20minutos    POLITICA     es     negative   Irán rechaza el plan de Trump para poner
2   20minutos    POLITICA     es     neutral    EEUU trabaja para lograr una reunión ent
3   20minutos    OTROS        es     neutral    El gráfico que muestra laenorme y sospec
4   20minutos    OTROS        es     negative   Sánchez se reivindica frente a Aznar por
5   20minutos    OTROS        es     negative   Los audios del momento del histórico apa
6   20minutos    CULTURA      es     neutral    Elduro testimonio de la joven Noelia Cas
7   20minutos    OTROS        es     negative   La jovenrecibirá este ju

In [23]:
# Actualizar todas_las_noticias con las versiones analizadas
# (El País ya estaba analizado en paso 3, las adicionales ahora también)
todas_las_noticias = noticias_analizadas.copy()  # El País (paso 3)
todas_las_noticias += noticias_adicionales_analizadas

print(f"✓ todas_las_noticias actualizado: {len(todas_las_noticias)} noticias (todas con análisis de Azure Language)")

from collections import Counter
conteo = Counter(n.get("fuente", "elpais") for n in todas_las_noticias)
print("\nPor fuente:")
for fuente, cantidad in conteo.most_common():
    print(f"  {fuente:<15} → {cantidad} noticias")

✓ todas_las_noticias actualizado: 33 noticias (todas con análisis de Azure Language)

Por fuente:
  elpais          → 10 noticias
  20minutos       → 10 noticias
  newsdata        → 10 noticias
  newsapi         → 3 noticias


---

## Paso 7 — Azure Translator: traducción bajo demanda

Las noticias de **NewsData.io** llegan en inglés. Usamos **Azure Translator** (API REST v3.0) para traducirlas al español bajo demanda.

| Concepto | Detalle |
|---|---|
| Servicio | Azure Translator v3.0 |
| Endpoint | Se lee de `TRANSLATOR_ENDPOINT` en `.env` |
| Método | POST con JSON `[{"Text": "..."}]` |
| Headers | `Ocp-Apim-Subscription-Key`, `Ocp-Apim-Subscription-Region`, `X-ClientTraceId` (uuid) |
| Resultado | Campos `titulo_es` y `resumen_es` añadidos a la noticia |

El módulo `src/azure_translator.py` expone una única función:
- `traducir_noticia(noticia, key, region, endpoint)` — traduce título y resumen de una noticia, añade `titulo_es` y `resumen_es`

### 7.1 — Configurar credenciales del Translator

In [24]:
if not TRANSLATOR_KEY or "TU_" in TRANSLATOR_KEY:
    print("⚠️  TRANSLATOR_KEY no encontrada en .env")
else:
    print(f"TRANSLATOR_KEY:      ✓ cargada")
    print(f"TRANSLATOR_REGION:   {TRANSLATOR_REGION}")
    print(f"TRANSLATOR_ENDPOINT: {TRANSLATOR_ENDPOINT}")

TRANSLATOR_KEY:      ✓ cargada
TRANSLATOR_REGION:   eastus
TRANSLATOR_ENDPOINT: https://api.cognitive.microsofttranslator.com/


### 7.2 — Traducir una noticia de prueba

Probamos `traducir_noticia` con una noticia ficticia en inglés antes de usarla con noticias reales.

In [25]:
import importlib

# Invalidar caché del finder para que detecte módulos nuevos en disco
importlib.invalidate_caches()

from src.azure_translator import traducir_noticia

# Noticia ficticia de prueba
noticia_prueba = {
    "titulo": "World leaders meet to discuss climate change policies",
    "resumen": "Technology stocks surge as AI investments grow rapidly across global markets",
}

print("Traduciendo noticia de prueba...\n")
traducir_noticia(noticia_prueba, TRANSLATOR_KEY, TRANSLATOR_REGION, TRANSLATOR_ENDPOINT)

print(f"  EN titulo:  {noticia_prueba['titulo']}")
print(f"  ES titulo:  {noticia_prueba['titulo_es']}")
print()
print(f"  EN resumen: {noticia_prueba['resumen']}")
print(f"  ES resumen: {noticia_prueba['resumen_es']}")
print()
print("✓ traducir_noticia funciona correctamente")

Traduciendo noticia de prueba...

  EN titulo:  World leaders meet to discuss climate change policies
  ES titulo:  Los líderes mundiales se reúnen para debatir políticas sobre cambio climático

  EN resumen: Technology stocks surge as AI investments grow rapidly across global markets
  ES resumen: Las acciones tecnológicas se disparan a medida que las inversiones en IA crecen rápidamente en los mercados globales

✓ traducir_noticia funciona correctamente


### 7.3 — Traducir una noticia real de NewsData

Usamos `traducir_noticia` sobre una noticia en inglés del paso 6.2. La función añade los campos `titulo_es` y `resumen_es` sin modificar los originales.

In [26]:
# Usar la primera noticia de NewsData (inglés) del paso 6.2
if 'noticias_newsdata' in dir() and noticias_newsdata:
    noticia_en = noticias_newsdata[0].copy()

    print(f"Original (EN):")
    print(f"  Título:  {noticia_en['titulo']}")
    print(f"  Resumen: {noticia_en['resumen'][:120]}...")
    print()

    traducir_noticia(noticia_en, TRANSLATOR_KEY, TRANSLATOR_REGION, TRANSLATOR_ENDPOINT)

    print(f"Traducida (ES):")
    print(f"  titulo_es:  {noticia_en['titulo_es']}")
    print(f"  resumen_es: {noticia_en['resumen_es'][:120]}...")
    print()
    print("✓ Campos añadidos: titulo_es, resumen_es")
else:
    print("⚠️  Ejecuta primero el paso 6.2 para tener noticias de NewsData")

Original (EN):
  Título:  CTA Construction Managers Awarded Contract to Build New Burlington Police Station
  Resumen: The new facility will replace the existing police station, which was originally constructed in the 1890s. BURLINGTON, Ma...

Traducida (ES):
  titulo_es:  Los responsables de construcción de la CTA adjudican el contrato para construir la nueva comisaría de policía de Burlington
  resumen_es: La nueva instalación reemplazará a la comisaría de policía existente, que fue construida originalmente en la década de 1...

✓ Campos añadidos: titulo_es, resumen_es


### 7.4 — Traducir todas las noticias en inglés

Recorremos todas las noticias recopiladas, detectamos las que están en inglés (por su campo `fuente`) y las traducimos en lote.

In [28]:
# Filtrar noticias en inglés (fuente newsdata)
noticias_en = [n for n in todas_las_noticias if n.get("fuente") == "newsdata"]

print(f"Noticias en inglés a traducir: {len(noticias_en)}\n")

traducidas = 0
for n in noticias_en:
    traducir_noticia(n, TRANSLATOR_KEY, TRANSLATOR_REGION, TRANSLATOR_ENDPOINT)
    traducidas += 1
    print(f"  ✓ {n['titulo'][:50]}...")
    print(f"    → {n['titulo_es'][:50]}...")
    print()

print(f"\n✓ {traducidas} noticias traducidas de EN → ES")

Noticias en inglés a traducir: 10

  ✓ CTA Construction Managers Awarded Contract to Buil...
    → Los responsables de construcción de la CTA adjudic...

  ✓ Better Industrial Stock: Tesla vs. Rocket Lab...
    → Mejor material industrial: Tesla vs. Rocket Lab...

  ✓ How Vingroup and VinFast Are Building a Green-Tech...
    → Cómo Vingroup y VinFast están construyendo un ecos...

  ✓ 2027: I will help Peter Obi get LP ticket if he lo...
    → 2027: Ayudaré a Peter Obi a conseguir el billete d...

  ✓ Bus service to divert route due to road closure...
    → Servicio de autobús desviado de ruta debido al cie...

  ✓ News24 | SA can find the money to cut fuel levies ...
    → Noticias24 | Sudáfrica puede encontrar el dinero p...

  ✓ News24 | National police commissioner Fannie Masem...
    → Noticias24 | La comisaria nacional de policía Fann...

  ✓ How Vingroup and VinFast Are Building a Green-Tech...
    → Cómo Vingroup y VinFast están construyendo un ecos...

  ✓ German Ifo Index Tak

### 7.5 — Ver estructura de una noticia traducida

Comprobamos que los campos `titulo_es` y `resumen_es` se han añadido correctamente junto a los campos originales.

In [29]:
import json

if noticias_en:
    ejemplo = noticias_en[0]
    print("Estructura completa de la noticia traducida:\n")
    print(json.dumps(ejemplo, ensure_ascii=False, indent=2))
else:
    print("⚠️  No hay noticias en inglés. Ejecuta los pasos 6.2 y 7.4 primero.")

Estructura completa de la noticia traducida:

{
  "titulo": "CTA Construction Managers Awarded Contract to Build New Burlington Police Station",
  "seccion": "economia",
  "resumen": "The new facility will replace the existing police station, which was originally constructed in the 1890s. BURLINGTON, Mass. , March 25, 2026 /PRNewswire/ -- CTA Construction Managers (CTA) has been awarded the contract for construction of the new Burlington Police Station for the Town of Burlington. The project marks a significant milestone in the Town's ongoing investment in modern, safe, and efficient municipal facilities. The new Burlington Police Station has been designed by Kaestle Boos Associates, with CHA serving as Owner's Project Manager (OPM). The new facility will replace the existing police station, which has served the community for more than 30 years and was originally constructed in the 1890s. \"While the existing historic building currently provides space for the BPD to operate, it does no

### 7.6 — Clasificar y guardar todas las noticias en carpetas

Ahora que todas las noticias (de las 4 fuentes) tienen `categoria` y `codigo_idioma` gracias al análisis de Azure Language (paso 6.5), las clasificamos en carpetas igual que hicimos en el paso 4 con El País.

In [30]:
from src.clasificador import clasificar_noticias

print("Clasificando noticias adicionales en carpetas...\n")
resultado_adicional = clasificar_noticias(noticias_adicionales_analizadas, base=BASE)

print(f"✓ {resultado_adicional['total']} artículos clasificados\n")
print("Distribución por carpeta:")
for carpeta, cantidad in sorted(resultado_adicional['distribucion'].items()):
    print(f"  {carpeta:<25} → {cantidad} artículo(s)")

Clasificando noticias adicionales en carpetas...

✓ 23 artículos clasificados

Distribución por carpeta:
  cultura/es                → 3 artículo(s)
  deportes/es               → 1 artículo(s)
  economia/en               → 6 artículo(s)
  otros/en                  → 3 artículo(s)
  otros/es                  → 7 artículo(s)
  politica/es               → 2 artículo(s)
  tecnologia/en             → 1 artículo(s)


### 7.7 — Inventario actualizado de todos los artículos

Verificamos el estado final de las carpetas con todas las noticias clasificadas (El País + fuentes adicionales).

In [31]:
from src.clasificador import listar_articulos

inventario = listar_articulos(base=BASE)

print("Inventario completo de artículos guardados:\n")
total = 0
for categoria, idiomas in inventario.items():
    for idioma, cantidad in idiomas.items():
        print(f"  [{categoria.upper():<12}] [{idioma}]  →  {cantidad} artículo(s)")
        total += cantidad

print(f"\n  Total: {total} artículo(s) en disco")

Inventario completo de artículos guardados:

  [CULTURA     ] [es]  →  3 artículo(s)
  [DEPORTES    ] [es]  →  10 artículo(s)
  [ECONOMIA    ] [en]  →  5 artículo(s)
  [ECONOMIA    ] [es]  →  4 artículo(s)
  [OTROS       ] [en]  →  3 artículo(s)
  [OTROS       ] [es]  →  26 artículo(s)
  [POLITICA    ] [es]  →  25 artículo(s)
  [TECNOLOGIA  ] [en]  →  1 artículo(s)
  [TECNOLOGIA  ] [es]  →  5 artículo(s)

  Total: 82 artículo(s) en disco


---

## Paso 8 — Podcast multi-fuente (español e inglés)

Ahora que tenemos noticias de **4 fuentes** (El País, 20 Minutos, NewsAPI, NewsData), generamos **dos podcasts** organizados por fuente:

| Podcast | Idioma | Voz | Incluye |
|---|---|---|---|
| `podcast_es_YYYY-MM-DD.mp3` | Español | `es-MX-Ximena:DragonHDLatestNeural` | Noticias en español + traducidas de NewsData |
| `podcast_en_YYYY-MM-DD.mp3` | Inglés | `en-US-AvaMultilingualNeural` | Noticias originales en inglés (NewsData) |

Cada podcast se organiza por bloques: primero la fuente, luego la categoría dentro de cada fuente.

### 8.1 — Previsualizar el podcast en español (multi-fuente)

Revisamos el texto antes de generar el audio. Ahora se organiza por fuente y luego por categoría.

In [32]:
import importlib
importlib.invalidate_caches()

# Recargar el módulo para usar las funciones nuevas
import src.azure_speech
importlib.reload(src.azure_speech)
from src.azure_speech import construir_texto_podcast_multifuente, generar_podcast_multifuente

# Previsualizar texto del podcast en español
texto_es = construir_texto_podcast_multifuente(todas_las_noticias, idioma="es")

print(f"Longitud del texto (ES): {len(texto_es)} caracteres\n")
print("-" * 60)
print(texto_es)

Longitud del texto (ES): 4175 caracteres

------------------------------------------------------------
Bienvenidos al resumen del día. Hoy es jueves 26 de March de 2026. Hemos recopilado noticias de 4 fuentes: El País, 20 Minutos, NewsData, NewsAPI. A continuación, las noticias más importantes. Pasamos ahora a las noticias de El País. En Política. La mayoría del Congreso rechaza la guerra pese a la bronca política. También, El último control a Montero en el Congreso anticipa la tensión de la campaña andaluza. Además, Meloni fuerza la dimisión de una ministra que se negaba a renunciar. Y para cerrar esta sección, El consejo de Indra se da una tregua y Escribano resiste el pulso a la SEPI. En Deportes. PP y Vox avanzan en un acuerdo programático para un Gobierno de coalición en Extremadura. En Tecnología. Meta y YouTube pierden en EE UU el juicio sobre la adicción de los menores a las redes sociales. En Otras noticias. Irán rechaza la propuesta de Trump para poner fin a la guerra y reivi

### 8.2 — Previsualizar el podcast en inglés (multi-fuente)

Texto del podcast con las noticias en inglés (solo de NewsData).

In [33]:
# Previsualizar texto del podcast en inglés
texto_en = construir_texto_podcast_multifuente(todas_las_noticias, idioma="en")

print(f"Longitud del texto (EN): {len(texto_en)} caracteres\n")
print("-" * 60)
print(texto_en)

Longitud del texto (EN): 1125 caracteres

------------------------------------------------------------
Welcome to today's news summary. Today is Thursday, March 26, 2026. We've gathered news from 1 sources: NewsData. Here are the most important headlines. Now, the news from NewsData. In Economy. CTA Construction Managers Awarded Contract to Build New Burlington Police Station. También, Better Industrial Stock: Tesla vs. Rocket Lab. Además, How Vingroup and VinFast Are Building a Green-Tech Ecosystem for Global Expansion. Igualmente, News24 | SA can find the money to cut fuel levies by 50% for 6 months – DA. Igualmente, German Ifo Index Takes A Nose Dive In March. Y para cerrar esta sección, CTA Construction Managers Awarded Contract to Build New Burlington Police Station. In Technology. How Vingroup and VinFast Are Building a Green-Tech Ecosystem for Global Expansion. In Other news. 2027: I will help Peter Obi get LP ticket if he loses in ADC – Datti Baba-Ahmed. También, Bus service to

### 8.3 — Generar podcast en español (multi-fuente)

In [38]:
if not SPEECH_KEY or "**_" in SPEECH_KEY:
    raise ValueError("Rellena SPEECH_KEY en el fichero .env antes de continuar")

print("Generando podcast en ESPAÑOL (multi-fuente)...\n")
ruta_audio_es = generar_podcast_multifuente(
    noticias=todas_las_noticias,
    speech_key=SPEECH_KEY,
    speech_region=SPEECH_REGION,
    idioma="es",
    carpeta_salida="../output"
)

print(f"\n✓ Podcast ES guardado en: {ruta_audio_es}")

Generando podcast en ESPAÑOL (multi-fuente)...

  → Construyendo texto del podcast (ES)...
  → Texto generado (4175 caracteres)
  → Convirtiendo a voz con Azure Speech...
  ✓ Audio generado: ..\output\podcast_es_2026-03-26.mp3

✓ Podcast ES guardado en: ..\output\podcast_es_2026-03-26.mp3


### 8.4 — Generar podcast en inglés (multi-fuente)

In [35]:
print("Generando podcast en INGLÉS (multi-fuente)...\n")
ruta_audio_en = generar_podcast_multifuente(
    noticias=todas_las_noticias,
    speech_key=SPEECH_KEY,
    speech_region=SPEECH_REGION,
    idioma="en",
    carpeta_salida="../output"
)

print(f"\n✓ Podcast EN guardado en: {ruta_audio_en}")

Generando podcast en INGLÉS (multi-fuente)...

  → Construyendo texto del podcast (EN)...
  → Texto generado (1125 caracteres)
  → Convirtiendo a voz con Azure Speech...
  ✓ Audio generado: ..\output\podcast_en_2026-03-26.mp3

✓ Podcast EN guardado en: ..\output\podcast_en_2026-03-26.mp3


### 8.5 — Reproducir los podcasts

In [37]:
import IPython.display as ipd

print("▶ Podcast en ESPAÑOL:")
ipd.display(ipd.Audio(ruta_audio_es))

print("\n▶ Podcast en INGLÉS:")
ipd.display(ipd.Audio(ruta_audio_en))

▶ Podcast en ESPAÑOL:



▶ Podcast en INGLÉS:
